# Адаптивные методы (20 баллов)

In [30]:
import os
from tqdm import tqdm
import numpy as np
import urllib.request
import zipfile
from IPython.display import Audio, display
import random
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
sns.set()
%config InlineBackend.figure_format = 'retina'

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim.optimizer import Optimizer

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Основная часть (10 баллов)


__Задача 1.__ В данном задании мы рассмотрим применение глубоких нейронных сетей для задачи шумоподавления аудиосигналов на датасете [`VoiceBank-DEMAND`](https://huggingface.co/datasets/JacobLinCool/VoiceBank-DEMAND-16k). Скачать можно с [диска](https://disk.yandex.ru/d/r0XIWHFBSxbiUA), возможно так будет быстрее, чем через код. Поскольку обработка аудиоданных требует сохранения временной структуры и тонких спектральных характеристик, мы будем использовать архитектуры на основе спектральных преобразований и автоэнкодеров. В нашем случае мы рассмотрим семейство адаптивных методов, таким как `AdaGrad`, `RMSProp`, `Adam`, `AdamW` и `Muon`.

In [5]:
# Создание директории для данных
os.makedirs("data", exist_ok=True)

# Скачивание данных
urls = [
    "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip",
    "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip",
    "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip",
    "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip"
]

for url in urls:
    filename = url.split('/')[-1]
    dest_path = f"data/{filename}"

    if os.path.exists(dest_path):
        print(f"Файл {filename} уже существует.")
    else:
        print(f"Скачивание {filename}...")

        with tqdm(unit='B', unit_scale=True, unit_divisor=1024, desc=filename) as pbar:
            def reporthook(block_num, block_size, total_size):
                if total_size > 0:
                    pbar.total = total_size
                pbar.update(block_size)

            urllib.request.urlretrieve(url, dest_path, reporthook)

    print(f"Распаковка {filename}...")
    with zipfile.ZipFile(dest_path, 'r') as zip_ref:
        zip_ref.extractall("data")

    os.remove(dest_path)
    print(f"{filename} скачан и распакован.\n")

Скачивание clean_trainset_28spk_wav.zip...


clean_trainset_28spk_wav.zip: 2.32GB [02:23, 17.3MB/s]                            


Распаковка clean_trainset_28spk_wav.zip...
clean_trainset_28spk_wav.zip скачан и распакован.

Скачивание noisy_trainset_28spk_wav.zip...


noisy_trainset_28spk_wav.zip: 2.64GB [05:05, 9.26MB/s]                            


Распаковка noisy_trainset_28spk_wav.zip...
noisy_trainset_28spk_wav.zip скачан и распакован.

Скачивание clean_testset_wav.zip...


clean_testset_wav.zip: 147MB [00:10, 14.6MB/s]                           


Распаковка clean_testset_wav.zip...
clean_testset_wav.zip скачан и распакован.

Скачивание noisy_testset_wav.zip...


noisy_testset_wav.zip: 163MB [00:15, 10.9MB/s]                           


Распаковка noisy_testset_wav.zip...
noisy_testset_wav.zip скачан и распакован.



В файле `model.py` реализованы все необходимые компоненты для обучения модели:  
- Датасет (`VoiceBankDataset`) — загрузка и предобработка аудио;
- Архитектура (`UNetSpectrogramDenoiser`) — U-Net для обработки спектрограмм;  
- Метрики ([`SNR`](https://en.wikipedia.org/wiki/Signal-to-noise_ratio), [`SI-SDR`](https://arxiv.org/abs/1811.02508)) – оценка качества очищенного звука;
- Трейнер (`trainer`).

In [9]:
url = "https://raw.githubusercontent.com/BRAIn-Lab-teaching/OPTIMIZATION-METHODS-COURSE/%D0%9F%D0%9C%D0%98_%D0%BE%D1%81%D0%B5%D0%BD%D1%8C_2025/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B8%D0%B5%20%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D1%8F/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B5%D0%B5%20%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5%2012/model.py"
!wget -O model.py "$url"

from model import VoiceBankDataset, UNetSpectrogramDenoiser, pad_and_crop_collate, trainer, extract_phase_and_chunk, istft_from_mag_phase

--2025-11-05 10:29:11--  https://raw.githubusercontent.com/BRAIn-Lab-teaching/OPTIMIZATION-METHODS-COURSE/%D0%9F%D0%9C%D0%98_%D0%BE%D1%81%D0%B5%D0%BD%D1%8C_2025/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B8%D0%B5%20%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D1%8F/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B5%D0%B5%20%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5%2012/model.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21955 (21K) [text/plain]
Saving to: ‘model.py’

model.py            100%[===================>]  21.44K  --.-KB/s    in 0.002s  

2025-11-05 10:29:11 (12.4 MB/s) - ‘model.py’ saved [21955/21955]



In [7]:
base = "./data"
train_noisy = os.path.join(base, "noisy_trainset_28spk_wav")
train_clean = os.path.join(base, "clean_trainset_28spk_wav")
test_noisy  = os.path.join(base, "noisy_testset_wav")
test_clean  = os.path.join(base, "clean_testset_wav")

full_train_ds = VoiceBankDataset(train_noisy, train_clean)
full_test_ds = VoiceBankDataset(test_noisy, test_clean)

train_ratio = 0.1
test_ratio = 0.1

# Разделяем датасеты
train_ds, _ = random_split(
    full_train_ds,
    [int(len(full_train_ds) * train_ratio), len(full_train_ds) - int(len(full_train_ds) * train_ratio)],
    generator=torch.Generator().manual_seed(420)
)

test_ds, _ = random_split(
    full_test_ds,
    [int(len(full_test_ds) * test_ratio), len(full_test_ds) - int(len(full_test_ds) * test_ratio)],
    generator=torch.Generator().manual_seed(420)
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4,
                          collate_fn=pad_and_crop_collate, drop_last=True
                         )
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=4,
                         collate_fn=pad_and_crop_collate, drop_last=False
                        )

print(f"Original train dataset size: {len(full_train_ds)}")
print(f"Original test dataset size: {len(full_test_ds)}")
print(f"Using train dataset size: {len(train_ds)}")
print(f"Using test dataset size: {len(test_ds)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Test batches per epoch: {len(test_loader)}")
nm, cm, nw, cw = next(iter(train_loader))
print("Noisy mag batch:", nm.shape)   # (B, 256, T_mag)
print("Noisy wav batch:", nw.shape)   # (B, T_wav)

Original train dataset size: 11572
Original test dataset size: 824
Using train dataset size: 1157
Using test dataset size: 82
Train batches per epoch: 72
Test batches per epoch: 6
Noisy mag batch: torch.Size([16, 256, 544])
Noisy wav batch: torch.Size([16, 284011])


__а) (1.5 балла)__ Перейдем к написанию оптимизационных методов. Реализуйте первый из адаптивных методов [`AdaGrad`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adagrad.html).

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальная точка $x^0 \in \mathbb{R}^d$
- Начальный буфер $G_{-1} = 0 \in \mathbb{R}^d$
- Размер шага $\{ \gamma_k \}_{k=0} > 0$
- Коэффициент $L_2$-регуляризации $\lambda \geq 0$

---

$k$_-ая итерация_:

1. Добавить регуляризационный член к градиенту:

$$
g^k = \nabla f \left(x^k\right) + \lambda x
$$

2. Обновить сумму квадратов градиентов:

$$
G^k = G^{k - 1} + \left(g^k\right)^2
$$

3. Сделать шаг алгоритма:

$$
x^{k + 1} = x^k - \gamma_k \frac{g^k}{\sqrt{G^k + \varepsilon}} $$

In [43]:
class Adagrad(Optimizer):
    """
    Реализация оптимизатора Adagrad.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        lr (float): Скорость обучения
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, lr=1e-2, weight_decay=1e-2):
        defaults = dict(lr=lr, weight_decay=weight_decay)
        super(Adagrad, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            lr = group['lr']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data

                if weight_decay != 0:
                    grad = grad.add(p.data, alpha=weight_decay)

                state = self.state[p]
                if len(state) == 0:
                    state['sum'] = torch.zeros_like(p.data)

                state['sum'].addcmul_(grad, grad, value=1)

                std = state['sum'].sqrt().add(1e-10)

                p.data.addcdiv_(grad, std, value=-lr)

        return loss

Проверьте работу:

In [44]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=Adagrad,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='Adagrad',
    test=True
)

Эпоха 1/1 | Время: 00:07 | Потери (обучение): 0.0172 | Потери (тест): 0.2320 | SNR: -0.19 дБ | SI-SDR: -2.78 дБ


__б) (1.5 балла)__ Теперь реализуем [`RMSProp`](https://docs.pytorch.org/docs/stable/generated/torch.optim.RMSprop.html). Основным отличием является, что теперь используется не сумма квадратов градиентов, а скользящая сумма.

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальная точка $x^0 \in \mathbb{R}^d$
- Начальный буфер $G_{-1} = 0 \in \mathbb{R}^d$
- Размер шага $\{ \gamma_k \}_{k=0} > 0$
- Коэффициент $L_2$-регуляризации $\lambda \geq 0$
- Коэффициент сглаживания квадратов градиентов $\beta \geq 0$

---

$k$_-ая итерация_:

1. Добавить регуляризационный член к градиенту:

$$
g^k = \nabla f \left(x^k\right) + \lambda x
$$

2. Обновить сумму квадратов градиентов:

$$
G^k = \beta G^{k - 1} + (1 - \beta) \left(g^k\right)^2
$$

3. Сделать шаг алгоритма:

$$
x^{k + 1} = x^k - \gamma_k \frac{g^k}{\sqrt{G^k + \varepsilon}} $$

In [45]:
class RMSprop(Optimizer):
    """
    Реализация оптимизатора RMSprop.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        lr (float): Скорость обучения
        beta (float): Параметр сглаживания квадратов градиентов
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, lr=1e-2, beta=0.99, weight_decay=1e-2):
        defaults = dict(lr=lr, beta=beta, weight_decay=weight_decay)
        super(RMSprop, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            lr = group['lr']
            beta = group['beta']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data

                if weight_decay != 0:
                    grad = grad.add(p.data, alpha=weight_decay)

                state = self.state[p]
                if len(state) == 0:
                    state['sum'] = torch.zeros_like(p.data)
                avg = state['sum']

                avg.mul_(beta).addcmul_(grad, grad, value=1 - beta)
                denom = avg.sqrt().add(1e-10)
                p.data.addcdiv_(grad, denom, value=-lr)

        return loss

Проверьте работу:

In [46]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=RMSprop,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='RMSprop',
    test=True
)

Эпоха 1/1 | Время: 00:09 | Потери (обучение): 0.0179 | Потери (тест): 0.1978 | SNR: -0.07 дБ | SI-SDR: -1.61 дБ


__в) (1.5 балла)__ Перейдем к самому популярному алгоритму оптимизации [`Adam`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html). В нем добавляется еще один момент для обновления сглаженного среднего градиентов.

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальная точка $x^0 \in \mathbb{R}^d$
- Начальный буфер $v^{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $G_{-1} = 0 \in \mathbb{R}^d$
- Размер шага $\{ \gamma_k \}_{k=0} > 0$
- Коэффициент $L_2$-регуляризации $\lambda \geq 0$
- Коэффициент сглаживания градиентов $\beta_1 \geq 0$
- Коэффициент сглаживания квадратов градиентов $\beta_2 \geq 0$

---

$k$_-ая итерация_:

1. Добавить регуляризационный член к градиенту:

$$
g^k = \nabla f \left(x^k\right) + \lambda x
$$

2. Обновить сумму градиентов:

$$
v^k = \beta_1 v^{k - 1} + (1 - \beta_1) g^k
$$

3. Выполним поправку смещения:

$$
\hat{v}^k = \frac{v^k}{1 - \beta_1^{k + 1}}
$$

4. Обновить сумму квадратов градиентов:

$$
G^k = \beta_2 G^{k - 1} + (1 - \beta_2) \left(g^k\right)^2
$$

5. Выполнить поправку смещения

$$
\hat{G}^k = \frac{G^k}{1 - \beta_2^{k + 1}}
$$

5. Сделать шаг алгоритма:

$$
x^{k + 1} = x^k - \gamma_k \frac{\hat{v}^k}{\sqrt{\hat{G}^k + \varepsilon}}
$$

In [41]:
class Adam(Optimizer):
    """
    Реализация оптимизатора Adam.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        lr (float): Скорость обучения
        betas (tuple): Параметры сглаживания
        weight_decay (float): Коэффициент L2-регуляризации
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super(Adam, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data

                if weight_decay != 0:
                    grad = grad.add(p.data, alpha=weight_decay)

                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['v'] = torch.zeros_like(p.data)
                    state['G'] = torch.zeros_like(p.data)

                v = state['v']
                G = state['G']
                state['step'] += 1

                v.mul_(beta1).add_(grad, alpha=1 - beta1)

                G.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                step = state['step']
                v_hat = v / (1 - beta1 ** step)
                G_hat = G / (1 - beta2 ** step)

                denom = G_hat.sqrt().add(1e-10)
                p.data.addcdiv_(v_hat, denom, value=-lr)

        return loss

Проверьте работу:

In [42]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=Adam,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='Adam',
    test=True
)

Эпоха 1/1 | Время: 00:08 | Потери (обучение): 0.0134 | Потери (тест): 0.1987 | SNR: -0.09 дБ | SI-SDR: -2.78 дБ


__г) (1.5 балла)__ Если задуматься, то регуляризация в `Adam` работает немного не так, как было изначально задумано, так как обновленный градиент после этого используется при подсчете моментов. При итоговом обновлении параметров мы получаем очень сложную зависимость от $\lambda$, что выливается в большую проблему при поиске оптимальных параметров. Поэтому было предложено использовать термин _затухание весов_, убрав регуляризационный член из обновления градиента:

$$
g^k = \nabla f \left(x^k\right) \color{red}{+ \lambda x}
$$

и добавить его при обновлении параметров:

$$
x^{k + 1} = x^k - \gamma_k \frac{\hat{v}^k}{\sqrt{\hat{G}^k + \varepsilon}} \color{green}{- \gamma_k \lambda x_k}
$$

Реализуйте алгоритм [`AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) с затуханием весов.

In [16]:
class AdamW(Optimizer):
    """
    Реализация оптимизатора AdamW.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        lr (float): Скорость обучения
        betas (tuple): Параметры сглаживания
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super(AdamW, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data

                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['v'] = torch.zeros_like(p.data)
                    state['G'] = torch.zeros_like(p.data)

                v = state['v']
                G = state['G']
                state['step'] += 1

                v.mul_(beta1).add_(grad, alpha=1 - beta1)

                G.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                step = state['step']
                v_hat = v / (1 - beta1 ** step)
                G_hat = G / (1 - beta2 ** step)

                denom = G_hat.sqrt().add(1e-10)

                step_adam = v_hat / denom

                if weight_decay != 0:
                    step_decay = weight_decay * p.data
                else:
                    step_decay = 0.0

                p.data.add_(step_adam + step_decay, alpha=-lr)


        return loss

Проверьте работу:

In [17]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=AdamW,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='AdamW',
    test=True
)

Эпоха 1/1 | Время: 00:07 | Потери (обучение): 0.0154 | Потери (тест): 0.2027 | SNR: -0.10 дБ | SI-SDR: -2.80 дБ


__д) (2 балла)__ Вспомним идею с предобуславливателями из дополнительной части домашней работы по методу Ньютона. Рассмотрим обновления метода `Adam`:

$$
x^{k + 1} = x^k - \gamma_k \frac{\hat{v}^k}{\sqrt{\hat{G}^k + \varepsilon}}
$$

Предобуславливатель это то, на что умножается градиент. `Adam` использует диагональный предобуславливатель $P_{\text{Adam}}$:

$$
P_{\text{Adam}} = \mathrm{diag} \left(\hat{G}^k\right)^{-\frac{1}{2}}.
$$

Но почему бы не использовать не диагональный предобуславливатель? Ведь это позволит уловить корелляции между градиентами различных параметров, что сильно улучшит понимание ландшафта функции потерь. Так и появился оптимизатор `Muon`. Он вместо диагонального предобуславливания использует матрицу ковариации, то есть

$$
P_{\text{Muon}} \approx \Sigma^{-\frac{1}{2}}.
$$

Но как же найти эту матрицу ковариации? Вспомним про алгоритм `Shampoo`, где использовались предобуславливатели и матрицы ковариаций считались как $G_k G_k^\top$ и $G_k^\top G_k$. Отбросим сейчас экспоненциальное сглаживание при обновлении этих матриц, тогда обновление алгоритма `Shampoo` выглядит так:

$$
X^{k + 1} = X^{k} - \gamma_k L_k G_k R_k = X^k - \gamma_k (G_k G_k^\top)^{-\frac{1}{4}} \cdot G_k \cdot (G_k^\top G_k)^{-\frac{1}{4}}.
$$

Применим SVD к матрице градиента: $G_k = U_k \Sigma_k V_k^\top$. Заметим также, что если воспользоваться свойством унитарных матриц, то получим:

$$
G_k G_k^\top = (U_k \Sigma_k V_k^\top) \cdot (U_k \Sigma_k V_k^\top)^\top = (U_k \Sigma_k V_k^\top) \cdot (V_k \Sigma_k U_k^\top) = U_k \Sigma_k^2 U_k^\top,
$$

$$
G_k^\top G_k = (U_k \Sigma_k V_k^\top)^\top \cdot (U_k \Sigma_k V_k^\top) = (V_k \Sigma_k U_k^\top) \cdot (U_k \Sigma_k V_k^\top) = V_k \Sigma_k^2 V_k^\top.
$$

Теперь нужно взять обратный корень 4 степени. Применим свойство унитарных матриц:

$$
X^{k + 1} = X^{k} - \gamma_k (U_k \Sigma_k^2 U_k^\top)^{-\frac{1}{4}} \cdot (U_k \Sigma_k V_k^\top) \cdot (V_k \Sigma_k^2 V_k^\top)^{-\frac{1}{4}} = X^{k} - \gamma_k (U_k \Sigma_k^{-\frac{1}{2}} U_k^\top) \cdot (U_k \Sigma_k V_k^\top) \cdot (V_k \Sigma_k^{-\frac{1}{2}} V_k^\top) = X^{k} - \gamma_k U_k V_k^\top.
$$

Теперь нам не нужно искать матрицу ковариаций, а достаточно лишь решить [Прокрустову задачу](https://en.wikipedia.org/wiki/Orthogonal_Procrustes_problem) (так называется приближение матрицы по Фробениусовой норме):

$$
\mathrm{Ortho(G)} = \arg \min_{\substack{OO^\top = I_d \\ \text{или} \\ O^\top O = I_d}} \| O - G \|_F  .
$$

Искать полное SVD разложение для решения этой задачи — долго и неприятно. Поэтому на практике используется итеративный алгоритм Ньютон-Шульца. Описывать его мы не будем, а реализацию уже приложим. Если есть желание, то можно прочитать про него в треде создателя `Muon` [здесь](https://kellerjordan.github.io/posts/muon/).

Таким образом предлагается заменить обновление в `Adam` с использованием метода Ньютон-Шульца для поиска ортогонального приближения матрицы:

$$
X^{k +1 } = X^k - \gamma_k \mathrm{Ortho}\left( \frac{\hat{V}^k}{\sqrt{\hat{G}^k + \varepsilon}} \right).
$$

_Примечание: Это не самый оптимальный способ реализации данного алгоритма. Желающие могут посмотреть на спидран pre-training of NanoGPT в этом [репозитории](https://github.com/KellerJordan/modded-nanogpt/blob/master/train_gpt.py)._

Реализуйте `Muon` для дальнейшего сравнения с другими адаптивными методами.

_Замечание: используйте регуляризацию, как в `AdamW`._

In [18]:
def zeropower_via_newtonschulz5(G, steps):
    """
    Ортогонализация матрицы методом Ньютона-Шульца.

    Параметры:
        G (torch.Tensor): Входная матрица (или батч матриц)
        steps (int): Количество итераций

    Возвращает:
        X (torch.Tensor): Ортогонализованная матрица
    """

    # Оптимальные коэффициенты для квинтической итерации
    a, b, c = (3.4445, -4.7750, 2.0315)

    X = G.bfloat16()

    # Транспонирование для "высоких" матриц
    if G.size(-2) > G.size(-1):
        X = X.mT

    # Нормировка по спектральной норме
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)

    # Итерации Ньютона-Шульца
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X

    # Обратное транспонирование при необходимости
    if G.size(-2) > G.size(-1):
        X = X.mT

    return X

In [25]:
class Muon(torch.optim.Optimizer):
    """
    Реализация оптимизатора Muon.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        lr (float): Скорость обучения
        betas (tuple): Параметры сглаживания
        weight_decay (float): Коэффициент L2-регуляризации
        ns_steps (int): Количество итераций Ньютона-Шульца
    """

    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), weight_decay=0, ns_steps=5):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay, ns_steps=ns_steps)
        super().__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            weight_decay = group['weight_decay']
            ns_steps = group['ns_steps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad.data

                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['v'] = torch.zeros_like(p.data)
                    state['G'] = torch.zeros_like(p.data)

                v = state['v']
                G = state['G']
                state['step'] += 1

                v.mul_(beta1).add_(grad, alpha=1 - beta1)
                G.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                step = state['step']
                v_hat = v / (1 - beta1 ** step)
                G_hat = G / (1 - beta2 ** step)

                denom = G_hat.sqrt().add_(1e-10)

                step_adam = v_hat / denom

                step_adam_2d = step_adam.unsqueeze(0)

                ortho_step_2d = zeropower_via_newtonschulz5(step_adam_2d, ns_steps)

                ortho_step = ortho_step_2d.squeeze(0)

                if weight_decay != 0:
                    step_decay = weight_decay * p.data
                else:
                    step_decay = 0.0

                p.data.add_(ortho_step + step_decay, alpha=-lr)

        return loss

Проверьте работу:

In [26]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=Muon,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='Muon',
    test=True
)

Эпоха 1/1 | Время: 00:09 | Потери (обучение): 0.0143 | Потери (тест): 0.2135 | SNR: -0.14 дБ | SI-SDR: -2.78 дБ


__е) (2 балла)__ Обучите модель. Поставьте параметр `lr=1e-3`, изменяйте только параметр `weight_decay in [1e-1, 1e-3, 1e-4]`. Число эпох поставьте равным 5.

In [27]:
weight_decays = [1e-1, 1e-3, 1e-4]
results = []

for wd in weight_decays:
    print(f"\n=== Training with weight_decay = {wd} ===")

    train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
        num_epochs=5,
        batch_size=16,
        model_class=UNetSpectrogramDenoiser,
        criterion=torch.nn.MSELoss(),
        optimizer_class=Muon,
        optimizer_params={'lr': 1e-3, 'weight_decay': wd},
        train_loader=train_loader,
        test_loader=test_loader,
        save_models=False,
        optimizer_name=f'Muon_wd_{wd}',
        test=True
    )

    results.append({
        'weight_decay': wd,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'test_snr': test_snr,
        'test_si_sdr': test_si_sdr
    })


=== Training with weight_decay = 0.1 ===


Эпоха 1/5 | Время: 00:10 | Потери (обучение): 0.0196 | Потери (тест): 0.2421 | SNR: -0.22 дБ | SI-SDR: -2.84 дБ


Эпоха 2/5 | Время: 00:08 | Потери (обучение): 0.0197 | Потери (тест): 0.2420 | SNR: -0.43 дБ | SI-SDR: -5.68 дБ


Эпоха 3/5 | Время: 00:07 | Потери (обучение): 0.0195 | Потери (тест): 0.2417 | SNR: -0.65 дБ | SI-SDR: -8.50 дБ


Эпоха 4/5 | Время: 00:09 | Потери (обучение): 0.0169 | Потери (тест): 0.2413 | SNR: -0.86 дБ | SI-SDR: -11.32 дБ


...

Эпоха 1/5 | Время: 00:09 | Потери (обучение): 0.0173 | Потери (тест): 0.2407 | SNR: -0.21 дБ | SI-SDR: -2.80 дБ


Эпоха 2/5 | Время: 00:09 | Потери (обучение): 0.0139 | Потери (тест): 0.2398 | SNR: -0.42 дБ | SI-SDR: -5.56 дБ


Эпоха 3/5 | Время: 00:07 | Потери (обучение): 0.0138 | Потери (тест): 0.2385 | SNR: -0.63 дБ | SI-SDR: -8.29 дБ


Эпоха 4/5 | Время: 00:09 | Потери (обучение): 0.0113 | Потери (тест): 0.2366 | SNR: -0.83 дБ | SI-SDR: -10.97 дБ


Эпоха 5/5 | Время: 00:07 | Потери (обучение): 0.0111 | Потери (тест): 0.2339 | SNR: -1.02 дБ | SI-SDR: -13.59 дБ


Постройте графики всех полученных метрик. Что можно сказать о сходимости методов? Влияет ли переосмысление `weight_decay` в случае `Adam` и `AdamW`?

Сравните оптимизаторы с помощью инференса на 3 случайных сэмплах из тестового датасета. Вам нужно загрузить предобученные модели и визуализировать сравнение зашумленного и очищенного аудио.

In [40]:
optimizers_to_evaluate = ['Adagrad', 'RMSprop', 'Adam', 'AdamW', 'Muon']
models = {}

# Загружаем сохраненные модели
print("Загрузка моделей...")
for opt_name in optimizers_to_evaluate:
    model = UNetSpectrogramDenoiser().to(device)
    model.load_state_dict(torch.load(f'models/{opt_name}_denoiser.pt')[opt_name])
    model.eval()
    models[opt_name] = model
print("Модели успешно загружены.")

# Выбираем 3 случайных сэмпла из тестового датасета
random_indices = random.sample(range(len(test_ds)), 3)

print("\nЗапуск инференса на 3 случайных сэмплах...")
for i, idx in enumerate(random_indices):
    noisy_mag, clean_mag, noisy_wav, clean_wav = test_ds[idx]

    print("\nЗашумленный сигнал:")
    display(Audio(noisy_wav.numpy(), rate=44100))

    print("\nЧистый сигнал:")
    display(Audio(clean_wav.numpy(), rate=44100))

    # Добавляем размерность батча
    noisy_mag_batch = noisy_mag.unsqueeze(0).to(device)

    print(f"\n--- Сэмпл #{i+1} ---")

    fig = plt.figure(figsize=(15, 6 + 2 * len(models)))
    gs = fig.add_gridspec(2 + len(models), 2)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_title("Зашумленное аудио")
    ax1.plot(noisy_wav.numpy())

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_title("Очищенное аудио")
    ax2.plot(clean_wav.numpy())

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_title("Зашумленная спектрограмма")
    ax3.imshow(noisy_mag.numpy(), aspect='auto', origin='lower')

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_title("Очищенная спектрограмма")
    ax4.imshow(clean_mag.numpy(), aspect='auto', origin='lower')

    row = 2
    for opt_name, model in models.items():
        with torch.no_grad():
            # Получаем предсказанную спектрограмму
            pred_mag_batch = model(noisy_mag_batch)

            # Извлекаем фазу и восстанавливаем аудио
            phase, noisy_chunk = extract_phase_and_chunk(
                noisy_wav.unsqueeze(0),
                pred_mag_batch.shape[1:],
                n_fft=512, hop_length=128,
                device=device
            )
            enhanced_wav_batch = istft_from_mag_phase(
                pred_mag_batch.squeeze(0), phase,
                n_fft=512, hop_length=128,
                length=noisy_chunk.shape[1]
            )
            enhanced_wav = enhanced_wav_batch.cpu().squeeze(0).numpy()
            print(f"\nОчищенный сигнал {opt_name}:")
            display(Audio(enhanced_wav, rate=44100))

            # Визуализация результата шумоподавления
            ax = fig.add_subplot(gs[row, :])
            ax.set_title(f"Результат шумоподавления с помощью {opt_name}")
            ax.plot(enhanced_wav)
            row += 1

    plt.tight_layout()
    plt.show()

Загрузка моделей...


FileNotFoundError: [Errno 2] No such file or directory: 'models/Adagrad_denoiser.pt'

## Дополнительная часть (10 баллов)

__Задача 2.__ Сложность достижения быстрой сходимости и качественных решений во многом зависит от выбранной скорости обучения. Приложения с большим количеством агентов, каждый из которых имеет свой оптимизатор, усложняют настройку скорости обучения. Некоторые оптимизаторы, настраиваемые вручную, показывают хорошие результаты, но эти методы обычно требуют квалификации специалистов и трудоемкой работы. Поэтому в последние годы для оптимизации без изменения скорости обучения стали популярны "беспараметрические" методы адаптивной скорости обучения (_parameter-free methods_). В этой задаче мы познакомимся с основными из них.

__a) (2.5 балла)__ Первый метод, который будет предложен к рассмотрению — Distance over Gradients (`DoG`). Он был впервые предложен в статье ["DoG is SGD's Best Friend: A Parameter-Free Dynamic Step Size Schedule"](https://arxiv.org/pdf/2302.12022). Основная идея — основываясь на методе `AdaGrad` найти способ адаптивно искать параметр шага $D$. Оценка на $D$ зависит от начальной точки $x^0$ и точки глобального минимума $x^*$. Если мы будем считать, что наш алгоритм сходится к оптимуму, то можно считать расстояние на $k$-ом шаге:

$$
\| x^k - x^0 \|_2 \to \| x^* - x^0 \|_2.
$$

Отсюда, обозначив в качестве $d_k$ оценку на $D$ на $k$-ом шаге, мы можем оценить шаг на $k$-ом шаге:

$$
\gamma_k = \frac{d_k}{\sqrt{\sum \limits_{t = 1}^k \| g^t \|^2 + \varepsilon}} = \frac{\max \limits_{t \in \overline{1, k}}\| x^0 - x^t\|_2}{\sqrt{\sum \limits_{t = 1}^k \| g^t \|^2 + \varepsilon}}.
$$

Однако пока непонятно, почему метод называется `DoG`. Заметим, что $d_k$ может быть записано как

$$
d_k = \max \limits_{t \in \overline{1, k - 1}} \left\| \sum \limits_{\tau = 0}^t \gamma_{\tau} \nabla f(x^\tau) \right\|_2.
$$

_Замечание: $d_k$ можно пересчитать следующим образом: $d_k = \max (d_{k - 1}, \|x^k - x^0\|_2)$._

In [ ]:
class DoG(Optimizer):
    """
    Реализация оптимизатора DoG.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        d_0 (float): Начальный шаг
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, d_0=1e-5, weight_decay=0):
        defaults = dict(d_0=d_0, weight_decay=weight_decay)
        super(DoG, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            d_0 = group['d_0']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                # YOUR CODE HERE

        return loss

Проверьте работу:

In [ ]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=DoG,
    optimizer_params={'weight_decay': 0},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='DoG',
    test=True
)

__б) (2.5 балла)__ Рассмотрим метод [`COCOB`](https://arxiv.org/pdf/1705.07795). Он основан на схеме ставок на монету, когда на каждой итерации ставится определенная сумма денег на исход подбрасывания монеты таким образом, чтобы максимизировать общее богатство, находящееся в распоряжении. Франческо Орабона применяет ту же идею к оптимизации функции, где ставка соответствует размеру шага, сделанного вдоль оси независимой переменной. Общее богатство и результат броска монеты соответствуют точке оптимума функции и отрицательному субградиенту функции в точке ставки, соответственно. В каждом раунде ставится часть текущего общего богатства (точка оптимума). Стратегия ставок разработана таким образом, что общее богатство не становится отрицательным ни в одной точке, а доля поставленных денег в каждом раунде увеличивается до тех пор, пока исход броска монеты не станет постоянным.

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальная точка $x^0 \in \mathbb{R}^d$
- Начальный буфер $L^{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $G_{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $\theta^{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $R^{-1} = 0 \in \mathbb{R}^d$
- Константа прогрева $\alpha \geq 0$
- Коэффициент $L_2$-регуляризации $\lambda \geq 0$

---

$k$_-ая итерация_:

1. Добавить регуляризационный член к градиенту:

$$
g^k = \nabla f \left(x^k\right) + \lambda x
$$

2. Обновить максимум абсолютных градиентов по координатам:

$$
L^k = \max \left(L^{k-1}, \left|g^k\right|\right)
$$

3. Обновить сумму модулей градиента:

$$
G^k = G^{k-1} + \left|g^k\right|
$$

4. Накопить знакованный градиент:

$$
\theta^k = \theta^{k - 1} - g^k
$$

5. Обновить богатство, не давая ему уйти в минус:

$$
R^k = \max \left(R^{k - 1} - g^k \left(x^k - x^0\right), 0\right)
$$

5. Сделать шаг алгоритма:

$$
x^{k + 1} = x^0 + \frac{R^k + L^k}{\max \left(G^k, \alpha L^k \right) L^k} \operatorname{sign} \left(\theta^k\right)
$$

In [ ]:
class COCOB(Optimizer):
    """
    Реализация оптимизатора COCOB.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        d_0 (float): Начальный шаг
        alpha (float): Определяет количество итераций прогрева
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, alpha=100, weight_decay=1e-2):
        defaults = dict(alpha=alpha, weight_decay=weight_decay)
        super(COCOB, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            alpha = group['alpha']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                # YOUR CODE HERE

        return loss

Проверьте работу:

In [ ]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=COCOB,
    optimizer_params={'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='COCOB',
    test=True
)

__в) (2.5 балла)__ Еще одним _paramter-free_ методом будет `Prodigy`, представленный в работе [Prodigy: An Expeditiously Adaptive Parameter-Free Learner](https://arxiv.org/pdf/2306.06101).

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальная точка $x^0 \in \mathbb{R}^d$
- Начальный буфер $v^{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $G^{-1} = 0 \in \mathbb{R}^d$
- Начальный буфер $r^{-1} = 0 \in \mathbb{R}$
- Начальный буфер $s^{-1} = 0 \in \mathbb{R}^d$
- Начальный шаг $d_0 \geq 0$
- Коэффициент сглаживания градиентов $\beta_1 \geq 0$
- Коэффициент сглаживания квадратов градиентов $\beta_2 \geq 0$
- Коэффициент $L_2$-регуляризации $\lambda \geq 0$

---


$k$_-ая итерация_:

1. Добавить регуляризационный член к градиенту:

$$
g^k = \nabla f \left(x^k\right) + \lambda x
$$

2. Обновить сумму градиентов:

$$
v^{k + 1} = \beta_1 v^k + (1 - \beta_1) d_k g^k
$$

3. Обновить сумму градиентов:

$$
G^{k + 1} = \beta_2 G^k + (1 - \beta_2) d_k^2 \left(g^k\right)^2
$$

4. Обновить богатство:

$$
r^{k + 1} = \sqrt{\beta_2} r^k + (1 - \sqrt{\beta_2}) d_k^2 \langle g^k, x^0 - x^k \rangle
$$

5. Обновить накопленные направления:

$$
s^{k + 1} = \sqrt{\beta_2} s^k + (1 - \sqrt{\beta_2}) d_k^2 g^k
$$

6. Вычислить кандидата на новый масштаб шага:

$$
\hat{d}_{k + 1} = \frac{r^{k + 1}}{\|s^{k + 1}\|_1}
$$

7. Обновить масштаб шага:

$$
d_{k + 1} = \max(d_k, \hat{d}_{k + 1})
$$

8. Выполнить шаг обновления параметров:

$$
x^{k + 1} = x^k - d_k \frac{v^{k + 1}}{\sqrt{G^{k + 1}} + d_k \varepsilon}
$$

In [ ]:
class Prodigy(Optimizer):
    """
    Реализация оптимизатора Prodigy.

    Параметры:
        params (Iterable): Итерируемый объект параметров для оптимизации или словарь
        betas (tuple): Параметры сглаживания
        d_0 (float): Начальный шаг
        weight_decay (float): Коэффициент L2-регуляризации
    """

    def __init__(self, params, betas=(0.9, 0.999), d_0=1e-4, weight_decay=0):
        defaults = dict(betas=betas, d_0=d_0, weight_decay=weight_decay)
        super(Prodigy, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Выполняет один шаг оптимизатора.

        Параметры:
            closure (Сallable): Замыкание, которое пересчитывает модель и возвращает loss

        Возвращает:
            loss (float): Значение функции потерь
        """
        loss = closure() if closure is not None else None

        for group in self.param_groups:
            beta1, beta2 = group['betas']
            d_0 = group['d_0']
            weight_decay = group['weight_decay']
            sqrt_beta2 = beta2 ** 0.5

            for p in group['params']:
                if p.grad is None:
                    continue

                # YOUR CODE HERE

        return loss

Проверьте работу:

In [ ]:
train_losses, test_losses, test_snr, test_si_sdr, model = trainer(
    num_epochs=1,
    batch_size=16,
    model_class=UNetSpectrogramDenoiser,
    criterion=torch.nn.MSELoss(),
    optimizer_class=Prodigy,
    optimizer_params={'weight_decay': 1e-2},
    train_loader=train_loader,
    test_loader=test_loader,
    save_models=False,
    optimizer_name='Prodigy',
    test=True
)

__г) (2.5 балл)__ Теперь пришло время обучения. Сравните данные оптимизаторы с `Adam`. Число эпох поставьте равным 5.

In [ ]:
# Ваше решение (Code)

Постройте графики всех полученных метрик. Что можно сказать о сходимости методов?

In [ ]:
# Ваше решение (Code)

Сравните оптимизаторы с помощью инференса на 3 случайных сэмплах из тестового датасета. Вам нужно загрузить предобученные модели и визуализировать сравнение зашумленного и очищенного аудио.

In [ ]:
optimizers_to_evaluate = ['Adam', 'COCOB', 'DoG', 'Prodigy']
models = {}

# Загружаем сохраненные модели
print("Загрузка моделей...")
for opt_name in optimizers_to_evaluate:
    model = UNetSpectrogramDenoiser().to(device)
    model.load_state_dict(torch.load(f'models/{opt_name}_denoiser.pt')[opt_name])
    model.eval()
    models[opt_name] = model
print("Модели успешно загружены.")

# Выбираем 3 случайных сэмпла из тестового датасета
random_indices = random.sample(range(len(test_ds)), 3)

print("\nЗапуск инференса на 3 случайных сэмплах...")
for i, idx in enumerate(random_indices):
    noisy_mag, clean_mag, noisy_wav, clean_wav = test_ds[idx]

    print("\nЗашумленный сигнал:")
    display(Audio(noisy_wav.numpy(), rate=44100))

    print("\nЧистый сигнал:")
    display(Audio(clean_wav.numpy(), rate=44100))

    # Добавляем размерность батча
    noisy_mag_batch = noisy_mag.unsqueeze(0).to(device)

    print(f"\n--- Сэмпл #{i+1} ---")

    fig = plt.figure(figsize=(15, 6 + 2 * len(models)))
    gs = fig.add_gridspec(2 + len(models), 2)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_title("Зашумленное аудио")
    ax1.plot(noisy_wav.numpy())

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_title("Очищенное аудио")
    ax2.plot(clean_wav.numpy())

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_title("Зашумленная спектрограмма")
    ax3.imshow(noisy_mag.numpy(), aspect='auto', origin='lower')

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_title("Очищенная спектрограмма")
    ax4.imshow(clean_mag.numpy(), aspect='auto', origin='lower')

    row = 2
    for opt_name, model in models.items():
        with torch.no_grad():
            # Получаем предсказанную спектрограмму
            pred_mag_batch = model(noisy_mag_batch)

            # Извлекаем фазу и восстанавливаем аудио
            phase, noisy_chunk = extract_phase_and_chunk(
                noisy_wav.unsqueeze(0),
                pred_mag_batch.shape[1:],
                n_fft=512, hop_length=128,
                device=device
            )
            enhanced_wav_batch = istft_from_mag_phase(
                pred_mag_batch.squeeze(0), phase,
                n_fft=512, hop_length=128,
                length=noisy_chunk.shape[1]
            )
            enhanced_wav = enhanced_wav_batch.cpu().squeeze(0).numpy()
            print(f"\nОчищенный сигнал {opt_name}:")
            display(Audio(enhanced_wav, rate=44100))

            # Визуализация результата шумоподавления
            ax = fig.add_subplot(gs[row, :])
            ax.set_title(f"Результат шумоподавления с помощью {opt_name}")
            ax.plot(enhanced_wav)
            row += 1

    plt.tight_layout()
    plt.show()